In [39]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key=os.getenv('GROQ_API_KEY')
tavily_api_key=os.getenv('tavily_api_key')


In [40]:
model=ChatGroq(model='openai/gpt-oss-20b',groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020EF6D16D10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020EF59C8990>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [41]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [42]:
#normal agent 
agent=create_deep_agent(model=model)

agent2=create_deep_agent(
    model=model,
    backend=StateBackend()
)

In [43]:
results=agent2.invoke({
    'messages':[{'role':'user','content':('Create a file /notes/todo.txt with exactly this content :\n'
    "1.Record video\n2.Edit video\n3. Upload video\n"
    "Then tell me you've done it")}]
})
results

{'messages': [HumanMessage(content="Create a file /notes/todo.txt with exactly this content :\n1.Record video\n2.Edit video\n3. Upload video\nThen tell me you've done it", additional_kwargs={}, response_metadata={}, id='d82f5b0c-cdee-4c49-9035-285377ad0cd6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to create a file at /notes/todo.txt with specified content. We should use write_file tool. After writing, we should respond that it\'s done. The content lines: "1.Record video\\n2.Edit video\\n3. Upload video". Ensure formatting exactly as given. Let\'s create file.', 'tool_calls': [{'id': 'fc_ec5b4720-59e7-4fca-855a-454948449814', 'function': {'arguments': '{"content":"1.Record video\\n2.Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 1934, 'total_tokens': 2039, 'completion_time': 0.114102819, 'completion_tokens_details':

In [44]:
print("\n --Backend Check----------------------------------------------------------------")
files=results.get('files',{})

if files:
    print(f'This is the length of the files {len(files)}')

    for path,content in files.items():
        print(f"\n {path}\n{'-'*40}\n{content}")

else:
    print("No file found in state")


 --Backend Check----------------------------------------------------------------
This is the length of the files 1

 /notes/todo.txt
----------------------------------------
{'content': '1.Record video\n2.Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-10-07T16:41:00.372336+00:00', 'modified_at': '2026-10-07T16:41:00.372336+00:00'}


In [45]:
followup=agent2.invoke({
    "messages": results["messages"] + [
        {
            'role':'user',
            'content':'Read/notes/todo.txt back to me.'
        }
    ],
    "files":results.get('files',{}),
})

print('\n--- Read-back (same thread)-------')
print(followup['messages'][-1].content)


--- Read-back (same thread)-------
Here’s the content of `/notes/todo.txt`:

```
1.Record video
2.Edit video
3. Upload video
```


## FileSystemBackend

In [46]:
from deepagents.backends import FilesystemBackend
ROOT='.'

agent=create_deep_agent(model=model,backend=FilesystemBackend(root_dir=Root,virtual_mode=True))
print(f'Agent created with FilesystemBackend(root_dir={ROOT!r})')
print(f'Files Written by the agent Will appear on your Actual disk')

Agent created with FilesystemBackend(root_dir='.')
Files Written by the agent Will appear on your Actual disk


In [47]:
results = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Create a file /notes/todo.txt with exactly this content:\n"
                "1.Record video\n"
                "2.Edit video\n"
                "3. Upload video\n"
                "Then tell me you've done it."
            )
        }
    ]
})

print(results["messages"][-1].content)

✅ Created `/notes/todo.txt` with the requested content.


## Store backend 

In [49]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store=InMemoryStore()

agent=create_deep_agent(
    model=model,
    backend=StoreBackend(
        namespace=lambda rt :('demo-user',),
    ),
    store=store,
)

print("Agent created with StoreBackend + static namespace")

Agent created with StoreBackend + static namespace
